In [50]:
import uuid
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

In [51]:
# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

In [52]:
print("Loading, merging, and populating GeoPulse datasets...")

# 1. Generate sample data frames representing the 3 sources
NUM_PINGS = 1000
BASE_LAT, BASE_LON = 40.748817, -73.985428

device_ids = [f"dev_{str(uuid.uuid4())[:8]}" for _ in range(50)]
df_pings = pd.DataFrame([{
    "ping_id": f"px_{str(uuid.uuid4())[:8]}",
    "device_id": random.choice(device_ids),
    "timestamp": datetime(2026, 11, 18, random.randint(6, 22), random.randint(0, 59)),
    "latitude": round(BASE_LAT + np.random.normal(0, 0.012), 6),
    "longitude": round(BASE_LON + np.random.normal(0, 0.012), 6),
    "horizontal_accuracy": round(random.uniform(2.0, 15.0), 1)
} for _ in range(NUM_PINGS)])

df_stores = pd.DataFrame([
    {
        "store_id": "store_coffee_01",
        "store_name": "Store A - 5th Ave",
        "store_status": "EXISTING",
        "store_lat": 40.748500,
        "store_lon": -73.985700
    },
    {
        "store_id": "store_coffee_02",
        "store_name": "Store B - Proposed 6th Ave",
        "store_status": "PROPOSED",
        "store_lat": 40.751000,
        "store_lon": -73.988000
    }
])

df_zones = pd.DataFrame([
    {
        "zone_id": "zone_midtown_01",
        "zone_name": "Midtown South Commercial",
        "zone_type": "High-Density Commercial",
        "base_footfall_category": "Transit Hub & Office Corridor"
    },
    {
        "zone_id": "zone_flatiron_02",
        "zone_name": "Flatiron Retail District",
        "zone_type": "Mixed-Use Commercial",
        "base_footfall_category": "Shopping District"
    }
])



Loading, merging, and populating GeoPulse datasets...


In [53]:
# 2. Perform Spatial Enrichment & Always Assign Nearest Store
merged_rows = []
for _, ping in df_pings.iterrows():
    # Assign zoning attribute based on geographic coordinate threshold
    zone = df_zones.iloc[0].to_dict() if ping['latitude'] > 40.750 else df_zones.iloc[1].to_dict()

    # Always find the nearest store regardless of distance
    closest_store = None
    min_dist = float('inf')
    for _, store in df_stores.iterrows():
        dist = np.sqrt((ping['latitude'] - store['store_lat'])**2 + (ping['longitude'] - store['store_lon'])**2)
        if dist < min_dist:
            min_dist = dist
            closest_store = store

    distance_meters = round(min_dist * 111320, 1)
    is_in_catchment = min_dist <= 0.0045 # ~500m walking radius threshold

    merged_rows.append({
        "ping_id": ping['ping_id'],
        "device_id": ping['device_id'],
        "timestamp": ping['timestamp'],
        "latitude": ping['latitude'],
        "longitude": ping['longitude'],
        "horizontal_accuracy": ping['horizontal_accuracy'],
        "zone_id": zone['zone_id'],
        "zone_name": zone['zone_name'],
        "zone_type": zone['zone_type'],
        "base_footfall_category": zone['base_footfall_category'],
        "nearest_store_id": closest_store['store_id'],         # Populated for all 1000 rows
        "nearest_store_name": closest_store['store_name'],     # Populated for all 1000 rows
        "nearest_store_status": closest_store['store_status'], # Populated for all 1000 rows
        "distance_to_store_meters": distance_meters,
        "is_within_catchment": is_in_catchment
    })

# 3. Create Master DataFrame & Export
df_master = pd.DataFrame(merged_rows)
df_master.to_csv("geo_pulse_master_dataset.csv", index=False)

print("Merge and population completed!")
print(f" - Master dataset generated successfully with {len(df_master)} rows and {len(df_master.columns)} columns.")
print(f" - Saved to `geo_pulse_master_dataset.csv`")

Merge and population completed!
 - Master dataset generated successfully with 1000 rows and 15 columns.
 - Saved to `geo_pulse_master_dataset.csv`


In [54]:
# Preview the first 10 rows
print(df_master.head(10))

       ping_id     device_id           timestamp   latitude  longitude  \
0  px_73cfa3ce  dev_392be9bd 2026-11-18 09:01:00  40.754778 -73.987087   
1  px_a318df05  dev_f406d1e7 2026-11-18 13:08:00  40.756589 -73.967152   
2  px_1e2e441a  dev_fd7ed26f 2026-11-18 08:37:00  40.746007 -73.988238   
3  px_70d16514  dev_adfeffe4 2026-11-18 08:13:00  40.767768 -73.976219   
4  px_f0fabaa4  dev_31ff8a58 2026-11-18 06:35:00  40.743183 -73.978917   
5  px_38bb7563  dev_2cfc4ecf 2026-11-18 19:14:00  40.743256 -73.991017   
6  px_c489532e  dev_f3cd3aae 2026-11-18 06:48:00  40.751721 -74.008387   
7  px_d01fc261  dev_fd1047a9 2026-11-18 19:21:00  40.728118 -73.992175   
8  px_ee6bfac0  dev_e86e726b 2026-11-18 16:06:00  40.736663 -73.981657   
9  px_c45966bb  dev_3a2acd08 2026-11-18 17:54:00  40.737921 -74.002376   

   horizontal_accuracy           zone_id                 zone_name  \
0                 11.6   zone_midtown_01  Midtown South Commercial   
1                 11.6   zone_midtown_01  Mid

In [55]:
# Preview the last 10 rows
print(df_master.tail(10))

         ping_id     device_id           timestamp   latitude  longitude  \
990  px_b1763f1d  dev_296422a0 2026-11-18 11:10:00  40.737157 -74.001983   
991  px_c573ad41  dev_19661156 2026-11-18 10:02:00  40.741296 -73.975079   
992  px_ab7ea02f  dev_fd7ed26f 2026-11-18 19:12:00  40.760255 -73.979271   
993  px_d643ddcf  dev_392be9bd 2026-11-18 10:53:00  40.757518 -73.979234   
994  px_9b9e790c  dev_a01fadcb 2026-11-18 08:05:00  40.741119 -73.980245   
995  px_6ddcd938  dev_d54bcdfd 2026-11-18 16:54:00  40.758422 -73.976377   
996  px_7b2bb695  dev_5e9ddf94 2026-11-18 06:50:00  40.763084 -73.976928   
997  px_cc01a093  dev_d4552e80 2026-11-18 22:50:00  40.753034 -73.972586   
998  px_e5ac4ba9  dev_fcbe4f0e 2026-11-18 10:09:00  40.748499 -73.996010   
999  px_193f0fd0  dev_d4552e80 2026-11-18 09:59:00  40.746860 -73.994367   

     horizontal_accuracy           zone_id                 zone_name  \
990                 14.6  zone_flatiron_02  Flatiron Retail District   
991                